In [0]:
# ============================================================
# SILVER LAYER: Cleaning and deduplication - PRO VERSION
# ============================================================
from pyspark.sql.functions import col, current_timestamp, row_number
from pyspark.sql.window import Window

print("📥 Reading bronze_telemetry table...")
df_bronze = spark.read.table("bronze_telemetry")
rows_before = df_bronze.count()
print(f"✅ Rows in Bronze: {rows_before}")

# 1. DEDUPLICATION - Keep last ingested record per event_id
print("\n🔁 Deduplicating by event_id (last one wins)...")
window_spec = Window.partitionBy("event_id").orderBy(col("ingest_timestamp").desc())
df_dedup = df_bronze.withColumn("rn", row_number().over(window_spec)) \
                    .filter(col("rn") == 1).drop("rn")

rows_after_dedup = df_dedup.count()
print(f"Duplicates removed: {rows_before - rows_after_dedup}")

# 2. SAFE CAST - This prevents pipeline from breaking if battery_pct = 'hola'
# In prod, dirty data doesn't break the job, it just gets filtered out
print("\n🛡️ Applying safe cast for battery_pct...")
df_casted = df_dedup.withColumn(
    "battery_pct_int",
    col("battery_pct").cast("int")  # 'hola' -> null, so it doesn't break
)

# 3. QUALITY FILTERS
print("\n🧹 Applying quality rules...")
df_silver = df_casted.filter(
    (col("event_id").isNotNull()) &
    (col("device_id").isNotNull()) &
    (col("speed_kmh") >= 0) & (col("speed_kmh") <= 200) &
    (col("engine_temp_c") >= -50) & (col("engine_temp_c") <= 150) &
    (col("battery_pct_int") >= 0) & (col("battery_pct_int") <= 100) # filtered on safe column
).drop("battery_pct").withColumnRenamed("battery_pct_int", "battery_pct")

rows_final = df_silver.count()
print(f"✅ Rows after filters: {rows_final}")
print(f"🗑  Discarded by quality rules: {rows_after_dedup - rows_final}")

# 4. Audit column - When was this processed in Silver?
df_silver_final = df_silver.withColumn("silver_processed_at", current_timestamp())

print("\n📊 Silver preview:")
display(df_silver_final.limit(5))

# 5. Save as managed Delta table
print("\n💾 Saving to silver_telemetry...")
df_silver_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_telemetry")

print(f"\n✅ SUCCESS! {rows_before} → {rows_final} rows ({round(rows_final/rows_before*100,2)}% retained)")
print("I'll use this clean table for Gold aggregations.")

In [0]:
# Celda 2 para que quede igual que en Bronze
# display(spark.sql("DESCRIBE HISTORY silver_telemetry"))